In [ ]:
!pip install -q rapidfuzz transformers scikit-learn pandas

In [ ]:
from google.colab import files
import os, re
up = files.upload()
for name in list(up):                                  
    clean = re.sub(r'(\s\(\d+\)|_\d+)(?=\.[A-Za-z]+$)', '', name)
    if clean != name:
        os.replace(name, clean)
need = ['quranic_verses.json', 'six_hadith_books.json', 'islamic_unified_dataset.jsonl']
missing = [f for f in need if not os.path.exists(f)]
print('كل الملفات موجودة ✓' if not missing else f'ناقص: {missing}')

In [ ]:
import re
import unicodedata

DIACRITICS = re.compile(
    r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4'
    r'\u06E7\u06E8\u06EA-\u06ED\u0640]'
)
PUNCTUATION = re.compile(r'[،؛؟!،.,:;\'"()\[\]{}<>«»\-_/\\|@#$%^&*+=~`]')
MULTI_SPACE = re.compile(r'\s+')
ALEF_VARIANTS = re.compile(r'[أإآٱ]')

def normalize_arabic(text: str) -> str:
    if not text:
        return ''
    text = unicodedata.normalize('NFC', text)
    text = DIACRITICS.sub('', text)
    text = ALEF_VARIANTS.sub('ا', text)
    text = PUNCTUATION.sub(' ', text)
    return MULTI_SPACE.sub(' ', text).strip()


def normalize_strict(text: str) -> str:
    return normalize_arabic(text)


def normalize_lenient(text: str) -> str:
    return normalize_arabic(text).replace('ة', 'ه').replace('ى', 'ي')


def tokenize(text: str):
    return [t for t in text.split() if t]


AR_DIAC = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED\u0640]')


def norm_match(s):
    if s is None:
        return ''
    s = AR_DIAC.sub('', s)
    s = s.replace('أ', 'ا').replace('إ', 'ا').replace('آ', 'ا').replace('ٱ', 'ا')
    s = s.replace('ؤ', 'و').replace('ئ', 'ي').replace('ة', 'ه').replace('ى', 'ي')
    s = re.sub(r'[^\u0600-\u06FF\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


STOPWORDS = set("""من في على ان أن إن الى إلى عن مع ما لا لم لن قد و ثم أو او هو هي هم انت أنتم كان كانت يكون تكون قال قالت
هذا هذه ذلك تلك الذي التي الذين اللاتي اللائي كل بعض غير عند بين حتى إذا اذا لو لكن بل يا أيها ايها""".split())


def content_words(tokens):
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]


from dataclasses import dataclass, field


@dataclass
class VerifierConfig:
    quran_correct_threshold: float = 0.94
    quran_uncertain_low: float = 0.45
    quran_min_coverage: float = 0.40
    hadith_correct_threshold: float = 0.79
    hadith_uncertain_low: float = 0.30
    hadith_min_coverage: float = 0.70
    quran_top_k: int = 25
    hadith_top_k: int = 15
    hadith_retrieval_guard: float = 0.20


@dataclass
class CorrectorConfig:
    max_window: int = 8              
    hadith_top_k: int = 40
    quran_strong: float = 0.65
    min_full_ratio: float = 0.40  
    quran_low: float = 0.55
    hadith_strong: float = 1.01
    hadith_low: float = 0.45


@dataclass
class PipelineConfig:
    verifier: VerifierConfig = field(default_factory=VerifierConfig)
    corrector: CorrectorConfig = field(default_factory=CorrectorConfig)
    min_detection_conf: float = 0.60     
    verified_min_conf: float = 0.75      
    unsupported_min_conf: float = 0.70   
    unsupported_strength: float = 0.35   
    corpus_cache: str = ".cache/corpus_index.pkl"



import re
import unicodedata as ud


try:  
    from rapidfuzz.distance import Levenshtein as _RFL
except ImportError:  
    _RFL = None


def _lcs_length(a, b):
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    if m < n:
        a, b, m, n = b, a, n, m
    prev = [0] * (n + 1)
    for i in range(m):
        curr = [0] * (n + 1)
        for j in range(n):
            curr[j + 1] = prev[j] + 1 if a[i] == b[j] else max(curr[j], prev[j + 1])
        prev = curr
    return prev[n]


def _levenshtein(a, b, max_len=600):
    a, b = a[:max_len], b[:max_len]
    if a == b:
        return 1.0
    la, lb = len(a), len(b)
    if la == 0 or lb == 0:
        return 0.0
    if _RFL is not None:
        return float(_RFL.normalized_similarity(a, b))
    prev = list(range(lb + 1))
    for i, ca in enumerate(a):
        curr = [i + 1]
        for j, cb in enumerate(b):
            cost = 0 if ca == cb else 1
            curr.append(min(curr[j] + 1, prev[j + 1] + 1, prev[j] + cost))
        prev = curr
    return 1.0 - prev[lb] / max(la, lb)


def _light_norm(t):
    t = ud.normalize('NFC', t)
    t = re.sub(r'[،؛؟!.,:;\'"()\[\]{}<>«»\-_/\\|@#$%^&*+=~`\u0640]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()


def compute_signals(claim: str, candidate: str, content_type: str = 'Ayah') -> dict:
    is_quran = (content_type == 'Ayah')
    norm_fn = normalize_strict if is_quran else normalize_lenient
    nc, nk = norm_fn(claim), norm_fn(candidate)

    c_tok, k_tok = set(tokenize(nc)), set(tokenize(nk))
    if c_tok and k_tok:
        inter = c_tok & k_tok
        token_overlap = len(inter) / len(c_tok | k_tok)
        coverage = len(inter) / len(c_tok)
    else:
        token_overlap = coverage = 0.0

    c_list, k_list = tokenize(nc), tokenize(nk)
    lcs_ratio = _lcs_length(c_list, k_list) / len(c_list) if c_list else 0.0
    edit_sim = _levenshtein(nc, nk)

    c_chars, k_chars = set(nc.replace(' ', '')), set(nk.replace(' ', ''))
    char_overlap = (len(c_chars & k_chars) / len(c_chars | k_chars)) if (c_chars or k_chars) else 0.0

    nc_ns, nk_ns = nc.replace(' ', ''), nk.replace(' ', '')
    is_substring = int(nc_ns in nk_ns or nk_ns in nc_ns)

    diacritic_sim = _levenshtein(_light_norm(claim), _light_norm(candidate), max_len=800) if is_quran else edit_sim
    span_len = len(c_list)

    if is_quran:
        if span_len < 4:
            w = dict(coverage=0.35, diacritic_sim=0.30, lcs_ratio=0.15, token_overlap=0.10, char_overlap=0.05, edit_sim=0.05)
        else:
            w = dict(coverage=0.25, diacritic_sim=0.30, lcs_ratio=0.20, token_overlap=0.10, char_overlap=0.05, edit_sim=0.10)
        composite = (w['coverage'] * coverage + w['diacritic_sim'] * diacritic_sim + w['lcs_ratio'] * lcs_ratio +
                     w['token_overlap'] * token_overlap + w['char_overlap'] * char_overlap + w['edit_sim'] * edit_sim)
        if is_substring and diacritic_sim >= 0.60:
            composite = max(composite, 0.88)
        elif is_substring:
            composite = max(composite, 0.75)
    else:
        if span_len < 4:
            w = dict(coverage=0.40, lcs_ratio=0.20, token_overlap=0.20, char_overlap=0.10, edit_sim=0.10)
        else:
            w = dict(coverage=0.30, lcs_ratio=0.28, token_overlap=0.18, char_overlap=0.12, edit_sim=0.12)
        composite = (w['coverage'] * coverage + w['lcs_ratio'] * lcs_ratio + w['token_overlap'] * token_overlap +
                     w['char_overlap'] * char_overlap + w['edit_sim'] * edit_sim)
        if is_substring:
            composite = max(composite, 0.82)

    return {
        'token_overlap': round(token_overlap, 4), 'coverage': round(coverage, 4),
        'lcs_ratio': round(lcs_ratio, 4), 'edit_sim': round(edit_sim, 4),
        'diacritic_sim': round(diacritic_sim, 4), 'char_overlap': round(char_overlap, 4),
        'is_substring': is_substring, 'composite': round(composite, 4),
    }


def best_match_score(claim, candidates, content_type='Ayah'):
    best_score, best_cand = 0.0, None
    for cand in candidates:
        sigs = compute_signals(claim, cand.get('text', ''), content_type)
        if sigs['composite'] > best_score:
            best_score, best_cand = sigs['composite'], {**cand, 'signals': sigs}
    return best_score, best_cand


"""BIO character/token label helpers from the Subtask 1A pipeline (logic unchanged)."""
LABEL2ID = {"O": 0, "B-Ayah": 1, "I-Ayah": 2, "B-Hadith": 3, "I-Hadith": 4}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


def build_character_labels(response, annotations):
    labels = ["O"] * len(response)
    for ann in sorted(annotations, key=lambda a: (a["end"] - a["start"]), reverse=True):
        start, end = max(0, ann["start"]), min(ann["end"], len(response))
        if start >= end:
            continue
        tag = ann["label"]
        labels[start] = f"B-{tag}"
        for i in range(start + 1, end):
            labels[i] = f"I-{tag}"
    return labels


def char_labels_to_token_labels(offsets, char_labels):
    out = []
    for start, end in offsets:
        out.append(-100 if start == end else LABEL2ID[char_labels[start]])
    return out


def token_labels_to_char_spans(offsets, token_labels, response=None):
    spans, current = [], None
    for (start, end), lid in zip(offsets, token_labels):
        if lid == -100:
            continue
        name = ID2LABEL[lid]
        if name == "O":
            if current is not None:
                spans.append(current)
                current = None
            continue
        prefix, etype = name.split("-")
        if prefix == "B":
            if current is not None:
                spans.append(current)
            current = {"label": etype, "start": start, "end": end}
        else:
            if current is not None and current["label"] == etype:
                current["end"] = end
            else:
                if current is not None:
                    spans.append(current)
                current = {"label": etype, "start": start, "end": end}
    if current is not None:
        spans.append(current)
    return spans


def merge_adjacent_spans(spans, response=None, max_gap=1):
    if not spans:
        return []
    spans = sorted(spans, key=lambda s: s["start"])
    merged = [dict(spans[0])]
    for s in spans[1:]:
        last = merged[-1]
        gap = s["start"] - last["end"]
        if s["label"] == last["label"] and 0 <= gap <= max_gap:
            last["end"] = max(last["end"], s["end"])
        else:
            merged.append(dict(s))
    return merged


import re

SUKUN = '\u0652'
SHADDA = '\u0651'
FATHATAN = '\u064B'
TANWEEN = set('\u064B\u064C\u064D')
IDGHAM_AFTER_NOON = set('نمرل')
IDGHAM_AFTER_LAM = set('لر')
MARKER = re.compile(r'^\(\d+\)$')
DIAC = set('\u0610\u0611\u0612\u0613\u0614\u0615\u0616\u0617\u0618\u0619\u061A'
           '\u064B\u064C\u064D\u064E\u064F\u0650\u0651\u0652\u0670'
           '\u06D6\u06D7\u06D8\u06D9\u06DA\u06DB\u06DC\u06DF\u06E0\u06E1\u06E2\u06E3\u06E4'
           '\u06E7\u06E8\u06EA\u06EB\u06EC\u06ED')


def base_letters(word):
    return ''.join(c for c in word if c not in DIAC)


def insert_shadda(word):
    return word if not word else word[0] + SHADDA + word[1:]


def _ends_with_tanween(w):
    if not w:
        return False
    if w[-1] in TANWEEN:
        return True
    return len(w) >= 2 and w[-1] in 'اى' and w[-2] == FATHATAN


def apply_idgham(text, extended=True):
    words = text.split(' ')
    noon_set = IDGHAM_AFTER_NOON if extended else set('نم')
    i = 0
    while i < len(words):
        w = words[i]
        if MARKER.match(w) or not w:
            i += 1
            continue
        j = i + 1
        while j < len(words) and MARKER.match(words[j]):
            j += 1
        if j < len(words):
            b = base_letters(words[j])
            first = b[0] if b else ''
            if w.endswith('\u0646' + SUKUN) and first in noon_set:
                words[i] = w[:-1]
                words[j] = insert_shadda(words[j])
            elif _ends_with_tanween(w) and first in noon_set:
                words[j] = insert_shadda(words[j])
            elif w.endswith('\u0645' + SUKUN) and first == '\u0645':
                words[i] = w[:-1]
                words[j] = insert_shadda(words[j])
            elif extended and w.endswith('\u0644' + SUKUN) and first in IDGHAM_AFTER_LAM:
                words[i] = w[:-1]
                words[j] = insert_shadda(words[j])
        i += 1
    return ' '.join(words)


In [ ]:
import json
import math
import os
import pickle
import time
from collections import Counter, defaultdict



def _grams(text, n=4):
    return {text[i:i + n] for i in range(len(text) - n + 1)}


class SourceRetriever:
    def __init__(self, quran_path, hadith_path, cache_path=None, verbose=True):
        self.verbose = verbose
        sig = self._signature(quran_path, hadith_path)
        data = self._load_cache(cache_path, sig)
        if data is None:
            t0 = time.time()
            self._log('Building corpus index (first run only, result is cached) ...')
            data = self._build(quran_path, hadith_path)
            data['sig'] = sig
            self._save_cache(cache_path, data)
            self._log(f'  built in {time.time() - t0:.0f}s')
        self.__dict__.update({k: v for k, v in data.items() if k != 'sig'})
        n_q = len(self.quran)
        self.quran_idf = {w: math.log((n_q + 1) / (len(d) + 1)) + 1 for w, d in self.q_content_index.items()}
        n_h = len(self.hadith)
        self.hadith_idf = {w: math.log((n_h + 1) / (len(d) + 1)) + 1 for w, d in self.h_content_index.items()}
        self._log(f'Retriever ready: {n_q} ayahs, {n_h} hadith records.')

    def _build(self, quran_path, hadith_path):
        with open(quran_path, encoding='utf-8', errors='ignore') as f:
            qdata = json.load(f)
        quran, q_norm_match, q_all_index, q_content_index, q_wc = [], [], defaultdict(list), defaultdict(list), []
        by_surah = defaultdict(dict)
        for e in qdata:
            text = (e.get('ayah_text') or '').strip()
            if not text:
                continue
            i = len(quran)
            quran.append({'surah_id': e.get('surah_id'), 'surah_name': e.get('surah_name', ''),
                          'ayah_id': e.get('ayah_id'), 'text': text})
            by_surah[e.get('surah_id')][e.get('ayah_id')] = i
            q_norm_match.append(norm_match(text))
            toks = tokenize(normalize_strict(text))
            q_wc.append(len(toks))
            for w in set(toks):
                q_all_index[w].append(i)
            for w in set(content_words(q_norm_match[-1].split())):
                q_content_index[w].append(i)

        with open(hadith_path, encoding='utf-8', errors='ignore') as f:
            hdata = json.load(f)
        hadith, h_content_index = [], defaultdict(list)
        for e in hdata:
            if not e:
                continue
            matn, full = (e.get('Matn') or '').strip() or None, (e.get('hadithTxt') or '').strip() or None
            if not (matn or full):
                continue
            rec = {'hadithID': e.get('hadithID'), 'book': e.get('BookID'), 'title': e.get('title'),
                   'matn': matn, 'full': full,
                   'norm_matn': norm_match(matn) if matn else None,
                   'norm_full': norm_match(full) if full else None}
            i = len(hadith)
            hadith.append(rec)
            words = set()
            for k in ('norm_matn', 'norm_full'):
                if rec[k]:
                    words |= set(content_words(rec[k].split()))
            for w in words:
                h_content_index[w].append(i)
        return dict(quran=quran, quran_by_surah=dict(by_surah), q_norm_match=q_norm_match,
                    q_all_index=dict(q_all_index), q_content_index=dict(q_content_index), q_wc=q_wc,
                    hadith=hadith, h_content_index=dict(h_content_index))

    @staticmethod
    def _signature(*paths):
        return [(os.path.basename(p), os.path.getsize(p)) for p in paths]

    def _load_cache(self, path, sig):
        if path and os.path.exists(path):
            try:
                with open(path, 'rb') as f:
                    d = pickle.load(f)
                if d.get('sig') == sig:
                    self._log(f'Loaded corpus index from cache {path}')
                    return d
            except Exception:
                pass
        return None

    def _save_cache(self, path, data):
        if path:
            os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
            with open(path, 'wb') as f:
                pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

    def _log(self, m):
        if self.verbose:
            print(f'[RETRIEVAL] {m}', flush=True)

    def search_quran_ayahs(self, query, top_k=25):
        q_words = tokenize(normalize_strict(query))
        if not q_words:
            return []
        votes = defaultdict(int)
        for w in q_words:
            for i in self.q_all_index.get(w, ()):
                votes[i] += 1
        scored = []
        for i, v in votes.items():
            cov = v / len(q_words)
            prec = v / self.q_wc[i] if self.q_wc[i] else 0.0
            scored.append((i, 2 * cov * prec / (cov + prec) if cov + prec > 0 else 0.0))
        scored.sort(key=lambda x: x[1], reverse=True)
        out = []
        for i, s in scored[:top_k]:
            c = dict(self.quran[i])
            c.update(type='Quran', retrieval_score=s)
            out.append(c)
        return out

    def quran_seed_ayahs(self, query_norm_words, top_k=25):
        return [i for i, _ in self._vote(self.q_content_index, self.quran_idf, query_norm_words, top_k)]

    def search_hadith(self, query, top_k=15, pool=40):
        words = content_words(norm_match(query).split())
        if not words:
            return []
        q_grams = _grams(normalize_lenient(query))
        out = []
        for i, _ in self._vote(self.h_content_index, self.hadith_idf, words, pool):
            r = self.hadith[i]
            text = r['matn'] or r['full']
            d_grams = _grams(normalize_lenient(text))
            cos = (len(q_grams & d_grams) / ((len(q_grams) * len(d_grams)) ** 0.5)) if q_grams and d_grams else 0.0
            out.append({'type': 'Hadith', 'idx': i, 'hadithID': r['hadithID'], 'book': r['book'],
                        'title': r['title'], 'text': text, 'has_matn': bool(r['matn']),
                        'retrieval_score': cos})
        out.sort(key=lambda c: c['retrieval_score'], reverse=True)
        return out[:top_k]

    @staticmethod
    def _vote(index, idf, words, top_k):
        scores = Counter()
        for w in set(words):
            wt = idf.get(w, 0.0)
            if wt <= 0:
                continue
            for c in index.get(w, ()):
                scores[c] += wt
        return scores.most_common(top_k)


from dataclasses import dataclass, field
from typing import Optional



@dataclass
class Verification:
    verdict: str                      
    confidence: float
    best_score: float
    method: str
    source: Optional[dict] = None     
    n_candidates: int = 0
    retrieval_top: float = 0.0


class Verifier:
    def __init__(self, retriever, cfg: VerifierConfig = None):
        self.kb = retriever
        self.cfg = cfg or VerifierConfig()

    def verify(self, span_text: str, content_type: str) -> Verification:
        if not span_text or not span_text.strip():
            return self._r('Incorrect', 0.95, 0.0, None, 0, 'empty_span')
        if content_type == 'Ayah':
            return self._quran(span_text)
        if content_type == 'Hadith':
            return self._hadith(span_text)
        return self._r('Incorrect', 0.5, 0.0, None, 0, 'unknown_type')


    def _quran(self, span):
        c = self.cfg
        cands = self.kb.search_quran_ayahs(span, top_k=c.quran_top_k)
        if not cands:
            return self._r('Incorrect', 0.8, 0.0, None, 0, 'no_candidates')
        score, best = best_match_score(span, cands, 'Ayah')
        sigs = best.get('signals', {}) if best else {}
        cov, is_sub = sigs.get('coverage', 0.0), sigs.get('is_substring', 0)

        if is_sub and cov >= c.quran_min_coverage:
            return self._r('Correct', min(0.98, 0.85 + score * 0.15), score, best, len(cands), 'substring_match')
        if score >= c.quran_correct_threshold and cov >= c.quran_min_coverage:
            return self._r('Correct', min(0.95, 0.70 + score * 0.25), score, best, len(cands), 'threshold_pass')
        if score <= c.quran_uncertain_low:
            return self._r('Incorrect', min(0.95, 0.70 + (1 - score) * 0.25), score, best, len(cands), 'threshold_fail')

        hi = 0
        for cand in cands[:10]:
            s = compute_signals(span, cand.get('text', ''), 'Ayah')
            if s['coverage'] >= 0.80 and s['lcs_ratio'] >= 0.75:
                hi += 1
        if hi >= 2:
            return self._r('Correct', 0.60 + min(0.20, hi * 0.05), score, best, len(cands), 'borderline_multi_cov')
        return self._r('Incorrect', 0.58, score, best, len(cands), 'borderline_default')


    def _hadith(self, span):
        c = self.cfg
        cands = self.kb.search_hadith(span, top_k=c.hadith_top_k)
        if not cands:
            return self._r('Incorrect', 0.75, 0.0, None, 0, 'no_candidates')
        top_ret = cands[0].get('retrieval_score', 0.0)
        score, best = best_match_score(span, cands, 'Hadith')
        sigs = best.get('signals', {}) if best else {}
        cov, is_sub = sigs.get('coverage', 0.0), sigs.get('is_substring', 0)

        if is_sub and cov >= c.hadith_min_coverage and top_ret >= c.hadith_retrieval_guard:
            return self._r('Correct', min(0.97, 0.80 + score * 0.17), score, best, len(cands), 'substring_match', top_ret)
        if score >= c.hadith_correct_threshold and cov >= c.hadith_min_coverage:
            return self._r('Correct', min(0.92, 0.65 + score * 0.27), score, best, len(cands), 'threshold_pass', top_ret)
        if score <= c.hadith_uncertain_low:
            return self._r('Incorrect', min(0.90, 0.65 + (1 - score) * 0.25), score, best, len(cands), 'threshold_fail', top_ret)

        moderate = 0
        for cand in cands[:8]:
            s = compute_signals(span, cand.get('text', ''), 'Hadith')
            if s['coverage'] >= 0.65 and s['lcs_ratio'] >= 0.55:
                moderate += 1
        if moderate >= 2 and top_ret >= 0.30:
            return self._r('Correct', 0.58 + min(0.22, moderate * 0.06), score, best, len(cands), 'borderline_multi_cov', top_ret)
        if top_ret < 0.25 or score < 0.45:
            return self._r('Incorrect', 0.60, score, best, len(cands), 'borderline_low_retrieval', top_ret)
        return self._r('Incorrect', 0.55, score, best, len(cands), 'borderline_default', top_ret)

    @staticmethod
    def _r(verdict, conf, score, best, n, method, top_ret=0.0):
        return Verification(verdict, round(conf, 4), round(score, 4), method, best, n, round(top_ret, 4))


import difflib
import re


_MARK = re.compile(r'^\(\d+\)$')


def compare(span_text: str, source_text: str) -> dict:
    a = span_text.split()
    b = [w for w in source_text.split() if not _MARK.match(w)]
    an = [norm_match(w) for w in a]
    bn = [norm_match(w) for w in b]
    sm = difflib.SequenceMatcher(None, an, bn, autojunk=False)
    blocks = [bl for bl in sm.get_matching_blocks() if bl.size > 0]
    if blocks and len(b) > 2 * len(a) + 10:          
        lo, hi = max(0, blocks[0].b - 3), min(len(b), blocks[-1].b + blocks[-1].size + 3)
        b, bn = b[lo:hi], bn[lo:hi]
        sm = difflib.SequenceMatcher(None, an, bn, autojunk=False)
    ops, missing, extra = [], [], []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        ops.append({'op': tag, 'span': ' '.join(a[i1:i2]), 'source': ' '.join(b[j1:j2])})
        if tag in ('delete', 'replace'):
            extra += a[i1:i2]
        if tag in ('insert', 'replace'):
            missing += b[j1:j2]
    return {'word_similarity': round(sm.ratio(), 3), 'word_diff': ops,
            'missing_from_span': missing, 'extra_in_span': extra, 'source_excerpt': ' '.join(b)}


import difflib
from dataclasses import dataclass
from typing import Optional



@dataclass
class CorrectionMatch:
    kind: str                  
    strength: float            
    full_ratio: float
    text: str                  
    source: dict               
    display: str = ''          


class Corrector:
    def __init__(self, retriever, cfg: CorrectorConfig = None):
        self.kb = retriever
        self.cfg = cfg or CorrectorConfig()

    def match_quran(self, query_text) -> Optional[CorrectionMatch]:
        kb = self.kb
        qn = norm_match(query_text)
        qwords = content_words(qn.split())
        if not qwords:
            return None
        qlen = len(qn)
        seeds = kb.quran_seed_ayahs(qwords, top_k=25)
        memo, best = {}, None        
        for si in seeds:
            surah, ayah = kb.quran[si]['surah_id'], kb.quran[si]['ayah_id']
            ayahs = kb.quran_by_surah[surah]
            min_a, max_a = min(ayahs), max(ayahs)
            for off in range(3):
                start = ayah - off
                if start < min_a:
                    continue
                wlen = -1
                for length in range(1, self.cfg.max_window + 1):
                    end = start + length - 1
                    if end > max_a:
                        break
                    wlen += len(kb.q_norm_match[ayahs[end]]) + 1
                    len_diff = abs(wlen - qlen)
                    ub = min(1.0, wlen / max(qlen, 1))
                    if best is not None:                       
                        bcov, bneg = best[0][0], best[0][1]
                        if ub < bcov or (ub == bcov and -len_diff < bneg):
                            continue
                    k = (surah, start, end)
                    if k in memo:
                        continue
                    window = ' '.join(kb.q_norm_match[ayahs[a]] for a in range(start, end + 1))
                    sm = difflib.SequenceMatcher(None, qn, window, autojunk=False)
                    matched = sum(b.size for b in sm.get_matching_blocks() if b.size >= 4)
                    cov = matched / max(qlen, 1)
                    key = (cov, -len_diff, sm.ratio())
                    memo[k] = key
                    if best is None or key > best[0]:
                        best = (key, cov, key[2], surah, start, end)
        if best is None:
            return None
        _, cov, ratio, surah, start, end = best
        display = ' '.join(kb.quran[kb.quran_by_surah[surah][a]]['text'] for a in range(start, end + 1))
        return CorrectionMatch(
            'Ayah', cov, ratio, self.ayah_text(surah, start, end),
            {'type': 'Quran', 'surah_id': surah, 'surah_name': kb.quran[kb.quran_by_surah[surah][start]]['surah_name'],
             'ayah_start': start, 'ayah_end': end}, display)

    def ayah_text(self, surah, start, end):
        kb, multi = self.kb, end > start
        parts = []
        for a in range(start, end + 1):
            t = kb.quran[kb.quran_by_surah[surah][a]]['text']
            parts.append(f'{t} ({a})' if multi else t)

        return apply_idgham(' '.join(parts)).replace('\u0640', '')

    def match_hadith(self, query_text) -> Optional[CorrectionMatch]:
        kb = self.kb
        qn = norm_match(query_text)
        qwords = content_words(qn.split())
        if not qwords:
            return None
        qlen = len(qn)
        best = None    
        for idx, _ in kb._vote(kb.h_content_index, kb.hadith_idf, qwords, self.cfg.hadith_top_k):
            rec = kb.hadith[idx]
            for field in ('norm_matn', 'norm_full'):
                text = rec[field]
                if not text:
                    continue
                ub = min(1.0, qlen / max(len(text), 1))
                if best is not None and ub < best[0][0]:
                    continue
                sm = difflib.SequenceMatcher(None, qn, text, autojunk=False)
                matched = sum(b.size for b in sm.get_matching_blocks() if b.size >= 4)
                cov, cand_cov = matched / max(qlen, 1), matched / max(len(text), 1)
                key = (min(cov, cand_cov), sm.ratio())
                if best is None or key > best[0]:
                    best = (key, idx, field, cov, cand_cov, key[1])
        if best is None:
            return None
        key, idx, field, cov, cand_cov, ratio = best
        rec = kb.hadith[idx]
        text = (rec['matn'] if field == 'norm_matn' else rec['full']).strip()
        return CorrectionMatch('Hadith', key[0], ratio, text,
                               {'type': 'Hadith', 'hadithID': rec['hadithID'], 'book': rec['book'],
                                'title': rec['title'], 'field': 'matn' if field == 'norm_matn' else 'full_text'}, text)

    def is_exact_ayah(self, span_text):

        kb, qn = self.kb, norm_match(span_text)
        if not qn:
            return False
        for si in kb.quran_seed_ayahs(content_words(qn.split()), 25):
            a = kb.quran[si]
            ays = kb.quran_by_surah[a['surah_id']]
            for off in range(4):
                st = a['ayah_id'] - off
                for ln in range(1, 6):
                    ids = [ays.get(x) for x in range(st, st + ln)]
                    if None in ids:
                        break
                    if qn in ' '.join(kb.q_norm_match[i] for i in ids):
                        return True
        return False

    def match(self, span_text, content_type):
        return self.match_quran(span_text) if content_type == 'Ayah' else self.match_hadith(span_text)


In [ ]:
import re
from dataclasses import dataclass
from typing import List, Optional


QUOTE_CHARS = ' \t\r\n"“”«»﴿﴾{}()[]'


@dataclass
class DetectedSpan:
    start: int
    end: int                       
    label: str                     
    confidence: Optional[float]    
    source: str                    
    text: str = ''


def trim_span(text, start, end):
    while start < end and text[start] in QUOTE_CHARS:
        start += 1
    while end > start and text[end - 1] in QUOTE_CHARS + '.،,؛:':
        end -= 1
    return start, end


class BertDetector:
    def __init__(self, model_dir, device=None, max_length=512, stride=256):
        import torch
        from transformers import AutoTokenizer, AutoModelForTokenClassification
        self.torch = torch
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForTokenClassification.from_pretrained(model_dir).to(self.device).eval()
        self.max_length, self.stride = max_length, stride

    def detect(self, text: str) -> List[DetectedSpan]:
        torch = self.torch
        if not text.strip():
            return []
        enc = self.tokenizer(text, return_offsets_mapping=True, truncation=False, return_tensors='pt')
        ids = enc['input_ids'][0]
        offsets = enc['offset_mapping'][0].tolist()
        total, n_lab = len(ids), len(LABEL2ID)
        acc, cnt = torch.zeros(total, n_lab), torch.zeros(total)
        start = 0
        with torch.no_grad():
            while start < total:
                end = min(start + self.max_length, total)
                w = ids[start:end].unsqueeze(0).to(self.device)
                logits = self.model(input_ids=w, attention_mask=torch.ones_like(w)).logits[0].cpu()
                acc[start:end] += logits
                cnt[start:end] += 1
                if end == total:
                    break
                start += self.stride
        acc /= cnt.unsqueeze(1).clamp(min=1)
        probs = torch.softmax(acc, dim=-1)
        pred = torch.argmax(acc, dim=-1).tolist()
        pred = [(-100 if a == b else p) for p, (a, b) in zip(pred, offsets)]   
        spans = merge_adjacent_spans(token_labels_to_char_spans(offsets, pred, text), text, max_gap=1)
        out = []
        for s in spans:
            a, b = trim_span(text, s['start'], s['end'])
            if b <= a:
                continue
            tok_idx = [i for i, (x, y) in enumerate(offsets) if y > x and x >= s['start'] and y <= s['end']]
            k = 'Ayah' if s['label'] == 'Ayah' else 'Hadith'
            ids_k = [LABEL2ID[f'B-{k}'], LABEL2ID[f'I-{k}']]
            conf = float(sum(probs[i, ids_k].sum() for i in tok_idx) / len(tok_idx)) if tok_idx else None
            out.append(DetectedSpan(a, b, k, conf, 'bert', text[a:b]))
        return out

AYAH_TRIGGERS = ['قال الله', 'قوله تعالى', 'قال تعالى', 'يقول الله', 'يقول تعالى', 'قال سبحانه', 'قوله سبحانه',
                 'في كتابه', 'سورة', 'الآية', 'الاية', 'الآيات', 'القرآن', 'القران', 'كتاب الله', 'عز وجل', 'جل جلاله',
                 'فقال تعالى', 'ذكر الله', '﴿']
HADITH_TRIGGERS = ['رسول الله', 'النبي', 'صلى الله عليه وسلم', 'ﷺ', 'عليه الصلاة والسلام', 'حديث', 'رواه', 'روى',
                   'متفق عليه', 'الحديث', 'فقال', 'قال ص', 'صلى الله عليه', 'وسلم']
FORMULA_WORDS = {norm_match(w) for w in 'قال قالت رسول الله صلى عليه وسلم النبي تعالى سبحانه عز وجل فقال يقول الكريم الشريف الحديث الآية روى رواه عن أن أنه البخاري ومسلم'.split()}

_PAIRS = [('“', '”'), ('«', '»'), ('﴿', '﴾'), ('{', '}'), ('(', ')'), ('[', ']')]


class RuleDetector:


    def __init__(self, retriever=None, min_words=3, context_chars=110, min_corpus_cov=0.6):
        self.kb, self.min_words, self.ctx, self.min_cov = retriever, min_words, context_chars, min_corpus_cov

    def _segments(self, text):
        segs = []
        quote_pos = [m.start() for m in re.finditer('"', text)]
        for a, b in zip(quote_pos, quote_pos[1:]):          
            segs.append((a + 1, b))
        for op, cl in _PAIRS:
            for m in re.finditer(re.escape(op) + r'(.*?)' + re.escape(cl), text, re.S):
                segs.append((m.start(1), m.end(1)))
        return segs

    @staticmethod
    def _trigger_type(context):
        best = (-1, None)
        for label, trig in (('Ayah', AYAH_TRIGGERS), ('Hadith', HADITH_TRIGGERS)):
            for t in trig:
                p = context.rfind(t)
                if p >= 0 and p + len(t) > best[0]:
                    best = (p + len(t), label)
        return best[1]

    def _corpus_cov(self, span):
        if self.kb is None:
            return 0.0, 0.0
        q_words = set(tokenize(normalize_strict(span)))
        if not q_words:
            return 0.0, 0.0
        qc = max((len(q_words & set(tokenize(normalize_strict(c['text'])))) / len(q_words)
                  for c in self.kb.search_quran_ayahs(span, top_k=5)), default=0.0)
        h_words = set(tokenize(normalize_lenient(span)))
        hc = max((len(h_words & set(tokenize(normalize_lenient(c['text'])))) / len(h_words)
                  for c in self.kb.search_hadith(span, top_k=5)), default=0.0) if h_words else 0.0
        return qc, hc

    def detect(self, text: str) -> List[DetectedSpan]:
        cands = []
        for s, e in self._segments(text):
            s, e = trim_span(text, s, e)
            if e <= s:
                continue
            inner = text[s:e]
            words = [w for w in norm_match(inner).split() if w]
            if len(words) < self.min_words or len(inner) > 3000:
                continue
            if sum(w in FORMULA_WORDS for w in words) / len(words) >= 0.6:
                continue
            trig = self._trigger_type(text[max(0, s - self.ctx):s])
            qc, hc = self._corpus_cov(inner)
            label = None
            if trig:
                label = trig
                other = hc if trig == 'Ayah' else qc
                mine = qc if trig == 'Ayah' else hc
                if other >= 0.8 and mine < 0.5:
                    label = 'Hadith' if trig == 'Ayah' else 'Ayah'
            elif max(qc, hc) >= self.min_cov and len(words) >= 4:
                label = 'Ayah' if qc >= hc else 'Hadith'
            if label is None:
                continue
            conf = max(qc, hc)
            cands.append((bool(trig), conf, e - s, s, e, label))
        cands.sort(key=lambda c: (c[0], c[1], c[2]), reverse=True)     
        taken = []
        for _, conf, _, s, e, label in cands:
            if all(e <= ts or s >= te for ts, te, _, _ in taken):
                taken.append((s, e, label, conf))
        taken.sort()
        return [DetectedSpan(s, e, l, None, 'rules', text[s:e]) for s, e, l, c in taken]


def build_detector(kind, retriever=None, model_dir=None, device=None, rules_use_corpus=False):
    if kind == 'rules':
        return RuleDetector(retriever if rules_use_corpus else None)
    if kind == 'bert':
        return BertDetector(model_dir, device=device)
    if model_dir:
        try:
            return BertDetector(model_dir, device=device)
        except Exception as exc:  
            print(f'[DETECTOR] could not load BERT model ({exc}); using rule-based detector')
    return RuleDetector(retriever if rules_use_corpus else None)


import time
from typing import List, Optional


STATUS_AR = {
    'VERIFIED': 'موثّق — النص مطابق للمصدر',
    'CORRECTED': 'مُصحَّح — استُبدل بنص المصدر (دليل قوي)',
    'UNSUPPORTED': 'غير مدعوم — لا يوجد مصدر مطابق (احتمال اختلاق)',
    'HUMAN_REVIEW': 'مراجعة بشرية — الدليل غير كافٍ أو غامض',
}


class IslamicContentVerifier:
    def __init__(self, quran_path, hadith_path, detector='rules', model_dir=None, cfg: PipelineConfig = None,
                 device=None, verbose=True):
        self.cfg = cfg or PipelineConfig()
        self.verbose = verbose
        self.retriever = SourceRetriever(quran_path, hadith_path, cache_path=self.cfg.corpus_cache, verbose=verbose)
        self.verifier = Verifier(self.retriever, self.cfg.verifier)
        self.corrector = Corrector(self.retriever, self.cfg.corrector)
        self.detector = build_detector(detector, self.retriever, model_dir, device)
        self.detector_name = type(self.detector).__name__

    def analyze(self, text: str) -> dict:
        t0 = time.time()
        spans = self.detector.detect(text)
        t_detect = time.time() - t0
        result = self._analyze_spans(text, spans)
        result['timings'] = {'detect_s': round(t_detect, 3), 'total_s': round(time.time() - t0, 3)}
        return result

    def analyze_spans(self, text: str, spans: List[dict]) -> dict:
        """Skip detection and use given spans [{label,start,end}] (used for evaluation / oracle mode)."""
        ds = [DetectedSpan(s['start'], s['end'], s['label'], None, 'given', text[s['start']:s['end']]) for s in spans]
        return self._analyze_spans(text, ds)

    def _analyze_spans(self, text, spans: List[DetectedSpan]) -> dict:
        reports = [self._process_span(i + 1, sp) for i, sp in enumerate(sorted(spans, key=lambda s: s.start))]
        counts = {k: 0 for k in STATUS_AR}
        for r in reports:
            counts[r['status']] += 1
        return {
            'input_text': text,
            'detector': self.detector_name,
            'spans': reports,
            'corrected_text': self._apply_corrections(text, reports),
            'summary': {'n_spans': len(reports), **counts, 'needs_human_review': counts['HUMAN_REVIEW'] > 0},
        }

    def _process_span(self, idx, sp: DetectedSpan) -> dict:
        cfg = self.cfg
        ver = self.verifier.verify(sp.text, sp.label)
        report = {
            'id': idx, 'type': sp.label, 'start': sp.start, 'end': sp.end, 'text': sp.text,
            'detection': {'backend': sp.source, 'confidence': None if sp.confidence is None else round(sp.confidence, 4)},
            'verification': {'verdict': ver.verdict, 'confidence': ver.confidence, 'score': ver.best_score,
                             'method': ver.method, 'n_candidates': ver.n_candidates},
            'evidence': self._evidence(sp, ver),
            'correction': None, 'suggestion': None,
        }

        status, reason = None, ''
        c = cfg.corrector

        def proposal_of(m):
            return None if m is None else {
                'text': m.text, 'display_text': m.display, 'source': m.source, 'match_strength': round(m.strength, 4),
                'full_ratio': round(m.full_ratio, 4), 'comparison': compare(sp.text, m.display)}

        if ver.verdict == 'Correct':
            exact = self.corrector.is_exact_ayah(sp.text) if sp.label == 'Ayah' else None
            report['verification']['exact_match'] = exact
            if ver.method.startswith('borderline') or ver.confidence < cfg.verified_min_conf:
                status, reason = 'HUMAN_REVIEW', 'weak_verification: matched a source but with low confidence'
            elif exact is False:
                status = 'HUMAN_REVIEW'
                reason = 'near_match: the quote is close to a source ayah but NOT identical (words missing / added / changed)'
                report['suggestion'] = proposal_of(self.corrector.match(sp.text, sp.label))
            else:
                status, reason = 'VERIFIED', f'matched source ({ver.method})'
        else:
            m = self.corrector.match(sp.text, sp.label)
            strong, low = (c.quran_strong, c.quran_low) if sp.label == 'Ayah' else (c.hadith_strong, c.hadith_low)
            proposal = proposal_of(m)
            if m is None or m.strength < low:
                if (m is None or m.strength < cfg.unsupported_strength) or (ver.confidence >= cfg.unsupported_min_conf and not ver.method.startswith('borderline')):
                    status, reason = 'UNSUPPORTED', 'no source in the corpus matches this quotation'
                else:
                    status, reason = 'HUMAN_REVIEW', 'insufficient_evidence: no clear source and low verification confidence'
                    report['suggestion'] = proposal
            elif m.strength >= strong and m.full_ratio >= c.min_full_ratio:
                status, reason = 'CORRECTED', f'strong match to {self._ref(m.source)} (strength {m.strength:.2f})'
                report['correction'] = {**proposal, 'applied': True}
            else:
                status = 'HUMAN_REVIEW'
                reason = (f'candidate source found ({self._ref(m.source)}, strength {m.strength:.2f}) '
                          f'but evidence is not strong enough for automatic correction')
                report['suggestion'] = proposal

        if (sp.confidence is not None and sp.confidence < cfg.min_detection_conf and status != 'HUMAN_REVIEW'):
            reason = f'low detection confidence ({sp.confidence:.2f}); was {status}: {reason}'
            status = 'HUMAN_REVIEW'
            if report['correction']:
                report['suggestion'], report['correction'] = {**report['correction'], 'applied': False}, None

        report['status'], report['status_ar'], report['reason'] = status, STATUS_AR[status], reason
        return report

    def _evidence(self, sp, ver):
        src = ver.source
        if not src:
            return None
        if sp.label == 'Ayah':
            ref = {'type': 'Quran', 'surah_id': src['surah_id'], 'surah_name': src['surah_name'], 'ayah': src['ayah_id']}
        else:
            ref = {'type': 'Hadith', 'hadithID': src['hadithID'], 'book': src['book'], 'title': src['title']}
        return {'source': ref, 'signals': src.get('signals'), 'comparison': compare(sp.text, src['text'])}

    @staticmethod
    def _ref(src):
        if src['type'] == 'Quran':
            a, b = src['ayah_start'], src['ayah_end']
            return f"Quran {src['surah_name']} {a}" + (f'-{b}' if b != a else '')
        return f"Hadith #{src['hadithID']}"

    @staticmethod
    def _apply_corrections(text, reports):
        out = text
        for r in sorted(reports, key=lambda r: r['start'], reverse=True):
            if r['status'] == 'CORRECTED' and r['correction'] and r['correction'].get('applied'):
                out = out[:r['start']] + r['correction']['display_text'] + out[r['end']:]
        return out

def render_report(result: dict, width: int = 100) -> str:
    S = result['summary']
    L = ['=' * width, f"تقرير التحقق  |  detector: {result['detector']}  |  spans: {S['n_spans']}", '=' * width]
    for r in result['spans']:
        L.append(f"\n[{r['id']}] {r['type']}  ({r['start']}-{r['end']})   الحالة: {r['status']}  — {r['status_ar']}")
        L.append(f"    النص المقتبس : {r['text'][:200]}")
        v = r['verification']
        d = r['detection']['confidence']
        L.append(f"    التحقق       : {v['verdict']}  ثقة={v['confidence']}  score={v['score']}  ({v['method']})"
                 + (f"  | ثقة الكشف={d}" if d is not None else ''))
        ev = r['evidence']
        if ev:
            s = ev['source']
            ref = (f"سورة {s['surah_name']} آية {s['ayah']}" if s['type'] == 'Quran' else f"حديث #{s['hadithID']} — {s['title']}")
            L.append(f"    المصدر       : {ref}")
            cmpd = ev['comparison']
            L.append(f"    المقارنة     : تشابه الكلمات={cmpd['word_similarity']}"
                     + (f" | ناقص: {' '.join(cmpd['missing_from_span'][:8])}" if cmpd['missing_from_span'] else '')
                     + (f" | زائد: {' '.join(cmpd['extra_in_span'][:8])}" if cmpd['extra_in_span'] else ''))
        for key, title in (('correction', 'التصحيح'), ('suggestion', 'اقتراح للمراجع')):
            c = r[key]
            if c:
                L.append(f"    {title:<12} : {c['display_text'][:200]}   [{IslamicContentVerifier._ref(c['source'])}, قوة={c['match_strength']}]")
        L.append(f"    السبب        : {r['reason']}")
    L += ['\n' + '-' * width, 'ملخص: ' + ' | '.join(f"{k}={S[k]}" for k in STATUS_AR),
          'يحتاج مراجعة بشرية: ' + ('نعم' if S['needs_human_review'] else 'لا'), '-' * width]
    if result['corrected_text'] != result['input_text']:
        L += ['\nالنص بعد التصحيح:', result['corrected_text']]
    return '\n'.join(L)


import csv
import re


def load_responses(xml_path):
    data = open(xml_path, encoding='utf-8').read()
    out = {}
    for b in re.findall(r'<Question>(.*?)</Question>', data, re.S):
        i, r = re.search(r'<ID>(.*?)</ID>', b, re.S), re.search(r'<Response>(.*?)</Response>', b, re.S)
        if i and r:
            out[i.group(1).strip()] = r.group(1)
    return out


def read_tsv(path, columns=None):
    with open(path, encoding='utf-8', newline='') as f:
        rows = list(csv.reader(f, delimiter='\t'))
    if not rows:
        return []
    if columns and rows[0][0] != 'Question_ID':
        return [dict(zip(columns, r)) for r in rows]
    header = rows[0]
    if 'Question_ID' not in header:
        raise ValueError(f'{path}: unexpected header {header}; expected a Question_ID column')
    return [dict(zip(header, r)) for r in rows[1:]]


def load_gold_spans_A(tsv_path):
    gold = {}
    for r in read_tsv(tsv_path):
        gold.setdefault(r['Question_ID'], [])
        if r['Label'] != 'NoAnnotation':
            gold[r['Question_ID']].append({'label': r['Label'], 'start': int(r['Span_Start']), 'end': int(r['Span_End'])})
    return gold


def load_B(tsv_path, xml_path):
    resp = load_responses(xml_path)
    out = []
    for r in read_tsv(tsv_path):
        s, e, lab = int(r['Span_Start']), int(r['Span_End']), r['Label']
        text = resp.get(r['Question_ID'], '')[s:e].strip() or r['Original_Span']
        out.append({'qid': r['Question_ID'], 'ann_id': r['Annotation_ID'], 'start': s, 'end': e,
                    'type': 'Ayah' if 'Ayah' in lab else 'Hadith',
                    'gt': 'Correct' if lab.startswith('Correct') else 'Incorrect', 'text': text})
    return out


def load_C(tsv_path, xml_path):
    resp = load_responses(xml_path)
    out = []
    for r in read_tsv(tsv_path):
        s, e = int(r['Span_Start']), int(r['Span_End'])
        text = resp.get(r['Question_ID'], '')[s:e] or r['Original_Span'].replace('...', ' ')
        out.append({'qid': r['Question_ID'], 'start': s, 'end': e, 'text': text,
                    'type': 'Ayah' if r['Label'] == 'WrongAyah' else 'Hadith',
                    'correction': r.get('Correction', '')})
    return out


import numpy as np

TAG = {'Ayah': 1, 'Hadith': 2}


def _macro_f1(truth, pred):
    labels = sorted(set(truth.tolist()) | set(pred.tolist()))
    f1s = []
    for l in labels:
        tp = int(((truth == l) & (pred == l)).sum())
        fp = int(((truth != l) & (pred == l)).sum())
        fn = int(((truth == l) & (pred != l)).sum())
        f1s.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f1s))


def detection_score(responses, gold, predicted):
    total, n = 0.0, 0
    for qid, g in gold.items():
        n += 1
        p = predicted.get(qid, [])
        if not g:
            total += 1.0 if not p else 0.0
            continue
        if not p:
            continue
        L = len(responses[qid])
        t_arr, p_arr = np.zeros(L, dtype=int), np.zeros(L, dtype=int)
        for s in g:
            t_arr[s['start']:s['end']] = TAG[s['label']]
        for s in p:
            p_arr[s['start']:s['end']] = TAG[s['label']]
        total += _macro_f1(t_arr, p_arr)
    return total / max(n, 1)


def align_spans(gold_spans, pred_spans, min_iou=0.5):
    pairs = []
    for gi, g in enumerate(gold_spans):
        for pi, p in enumerate(pred_spans):
            if g['label'] != p['label']:
                continue
            inter = max(0, min(g['end'], p['end']) - max(g['start'], p['start']))
            union = (g['end'] - g['start']) + (p['end'] - p['start']) - inter
            iou = inter / union if union else 0.0
            if iou >= min_iou:
                pairs.append((iou, gi, pi))
    pairs.sort(reverse=True)
    used_g, used_p, out = set(), set(), {}
    for iou, gi, pi in pairs:
        if gi in used_g or pi in used_p:
            continue
        used_g.add(gi); used_p.add(pi); out[gi] = pi
    return out


import json


def remove_default_diac(s):
    out = s
    for a, b in [("َا", "ا"), ("ِي", "ي"), ("ُو", "و"), ("الْ", "ال"), ("ْ", ""), ("اَ", "ا"),
                 ("اِ", "ا"), ("لِا", "لا"), ("اً", "ًا")]:
        out = out.replace(a, b)
    return out


class CorrectionScorer:
    def __init__(self, quran_path, hadith_path):
        q = json.load(open(quran_path, encoding='utf-8'))
        h = json.load(open(hadith_path, encoding='utf-8'))
        self.db = {remove_default_diac(i['ayah_text']) for i in q if i and i.get('ayah_text')}
        for i in h:
            if i:
                for k in ('hadithTxt', 'Matn'):
                    if i.get(k):
                        self.db.add(remove_default_diac(i[k]))

    def is_correct(self, pred, ref):
        p, r = remove_default_diac(pred), remove_default_diac(ref)
        return p == r or (r in p and p in self.db)


In [ ]:
import json
import random


def _span(s, e, label, text, verdict=None, correction=None, **extra):
    d = {'start': s, 'end': e, 'label': label, 'text': text, 'verdict': verdict, 'correction': correction}
    d.update(extra)
    return d


def build_real_records(data_dir='.'):
    j = lambda f: f'{data_dir}/{f}'
    recs = []
    respA, goldA = load_responses(j('dev_SubtaskA.xml')), load_gold_spans_A(j('dev_SubtaskA.tsv'))
    for q, t in respA.items():
        spans = [_span(g['start'], g['end'], g['label'], t[g['start']:g['end']]) for g in goldA.get(q, [])]
        recs.append({'id': q, 'source': 'real_A', 'text': t, 'spans': spans,
                     'complete': {'detection': True, 'verdict': False, 'correction': False}})

    respB, by = load_responses(j('dev_SubtaskB.xml')), {}
    for s in load_B(j('dev_SubtaskB.tsv'), j('dev_SubtaskB.xml')):
        by.setdefault(s['qid'], []).append(s)
    for q, t in respB.items():
        spans = [_span(s['start'], s['end'], s['type'], t[s['start']:s['end']], verdict=s['gt']) for s in by.get(q, [])]
        recs.append({'id': q, 'source': 'real_B', 'text': t, 'spans': spans,
                     'complete': {'detection': True, 'verdict': True, 'correction': False}})
    respC, by = load_responses(j('dev_SubtaskC.xml')), {}
    for s in load_C(j('dev_SubtaskC.tsv'), j('dev_SubtaskC.xml')):
        by.setdefault(s['qid'], []).append(s)
    for q, t in respC.items():
        spans = [_span(s['start'], s['end'], s['type'], t[s['start']:s['end']], verdict='Incorrect', correction=s['correction'])
                 for s in by.get(q, [])]
        recs.append({'id': q, 'source': 'real_C', 'text': t, 'spans': spans,
                     'complete': {'detection': False, 'verdict': False, 'correction': True}})
    return recs


TOPICS = ['الصبر عند المصائب', 'بر الوالدين', 'الصدقة', 'صلة الرحم', 'الصلاة في وقتها', 'الصيام', 'التوبة', 'الزواج',
          'العدل بين الأبناء', 'طلب العلم', 'الصدق في القول', 'حسن الجوار', 'الزكاة', 'الحج', 'حسن الخلق', 'التوكل على الله',
          'الأمانة', 'الوفاء بالعهد', 'غض البصر', 'الإحسان إلى اليتيم', 'ذكر الله', 'الدعاء', 'الشكر', 'الرفق بالحيوان',
          'النهي عن الغيبة', 'إصلاح ذات البين', 'الاستغفار', 'العمل والكسب الحلال', 'معاملة الجار', 'التعامل مع الغضب']
INTROS = ['بخصوص سؤالك عن {t}، فإن الإسلام أولى هذا الأمر عناية كبيرة.', 'يُعد موضوع {t} من المسائل المهمة في الشريعة.',
          'أما عن {t} فالأصل فيه ما جاء في الكتاب والسنة.', 'سؤال جيد حول {t}، وإليك التفصيل:',
          'لقد حث الإسلام على {t} في مواضع كثيرة، ومن ذلك ما يلي.']
AYAH_LEADS = ['قال الله تعالى:', 'وقد ورد في القرآن الكريم قوله تعالى:', 'يقول الله تعالى في كتابه العزيز:', 'والدليل على ذلك قوله تعالى:',
              'وقال سبحانه:', 'ومما جاء في القرآن الكريم:', 'وفي ذلك يقول الحق تبارك وتعالى:', 'كما في قوله عز وجل:']
HADITH_LEADS = ['وقال رسول الله صلى الله عليه وسلم:', 'وقد ورد في الحديث الشريف:', 'روى البخاري ومسلم عن النبي صلى الله عليه وسلم أنه قال:',
                'وعن ذلك قال النبي عليه الصلاة والسلام:', 'وفي الحديث الصحيح قال ﷺ:', 'ومن السنة النبوية قوله صلى الله عليه وسلم:',
                'جاء في سنن الترمذي أن النبي صلى الله عليه وسلم قال:', 'وفي الحديث الذي رواه أبو داود:']
TAILS = ['وهذا من الأمور التي حث عليها الإسلام بشدة.', 'وقد أجمع العلماء على أهمية هذا الأمر.', 'وفي هذا دلالة واضحة على رحمة الله بعباده.',
         'وهو من مكارم الأخلاق التي دعا إليها ديننا الحنيف.', 'وقد فسر العلماء هذا المعنى بتفصيل في كتبهم.', 'ويستفاد منه وجوب الالتزام بهذا الأمر.', '']
NEGATIVES = ['إن الحياة مليئة بالتحديات التي تستوجب الصبر والحكمة في التعامل معها بإيجابية.',
             'وينبغي على الإنسان أن يسعى دائما لتحقيق التوازن بين حقوق نفسه وحقوق من حوله.',
             'وقد بينت الدراسات الحديثة أهمية هذا السلوك في بناء المجتمعات المتماسكة.',
             'إن التربية السليمة تقوم على غرس القيم النبيلة في نفوس الأبناء منذ الصغر.',
             'قال أحد العلماء: "الصبر مفتاح الفرج وثمرة اليقين".', 'ويقال في الحكمة: "من جد وجد ومن زرع حصد".',
             'وقال بعض السلف: "ما أحسن الدنيا إذا أقبلت بالطاعة وما أجمل الآخرة إذا حضرت بالعمل".']
DIAC_SWAP = {'\u064E': '\u064F', '\u064F': '\u0650', '\u0650': '\u064E', '\u064B': '\u064C', '\u064C': '\u064D', '\u064D': '\u064B'}


class SyntheticBuilder:
    """Synthetic annotated answers built from the Quran/Hadith corpora. Labels are correct BY CONSTRUCTION
    (exact corpus text = Correct; corrupted = Incorrect + original as correction; half-and-half mix of two
    different texts = Incorrect, no source -> 'خطأ'). Use for TRAINING only, never for evaluation."""

    def __init__(self, retriever, seed=42):
        self.rng = random.Random(seed)
        self.kb = retriever
        wc = lambda s: len(s.split())
        self.ayahs = [i for i, a in enumerate(retriever.quran) if 5 <= wc(a['text']) <= 45]
        self.hadiths = [h['matn'] for h in retriever.hadith if h['matn'] and 6 <= wc(h['matn']) <= 50 and len(h['matn']) <= 380]

    def _pick(self, label):
        r = self.rng
        if label == 'Ayah':
            i = r.choice(self.ayahs)
            a = self.kb.quran[i]
            return {'text': a['text'], 'ref': f"سورة {a['surah_name']}: {a['ayah_id']}", 'multi': False}
        return {'text': r.choice(self.hadiths), 'ref': None, 'multi': False}

    def _multi_ayah(self):
        kb, r = self.kb, self.rng
        for _ in range(20):
            i = r.choice(self.ayahs)
            a = kb.quran[i]
            nxt = kb.quran_by_surah[a['surah_id']].get(a['ayah_id'] + 1)
            if nxt is not None and len(a['text'].split()) + len(kb.quran[nxt]['text'].split()) <= 60:
                return {'text': a['text'] + ' ' + kb.quran[nxt]['text'], 'ref': f"سورة {a['surah_name']}: {a['ayah_id']}-{a['ayah_id'] + 1}", 'multi': True}
        return self._pick('Ayah')

    def _corrupt(self, text, label):
        r, w = self.rng, text.split()
        kinds = ['delete_word', 'swap_words', 'replace_word'] + (['diacritic'] if label == 'Ayah' else [])
        r.shuffle(kinds)
        for k in kinds:
            if k == 'delete_word' and len(w) >= 4:
                x = w[:]; x.pop(r.randint(1, len(x) - 2)); return ' '.join(x), k
            if k == 'swap_words' and len(w) >= 3:
                x = w[:]; i = r.randint(0, len(x) - 2); x[i], x[i + 1] = x[i + 1], x[i]
                if x != w: return ' '.join(x), k
            if k == 'replace_word' and len(w) >= 3:
                pool = (self._pick(label)['text']).split()
                x = w[:]; x[r.randint(0, len(x) - 1)] = r.choice(pool)
                if x != w: return ' '.join(x), k
            if k == 'diacritic':
                pos = [i for i, c in enumerate(text) if c in DIAC_SWAP]
                if pos:
                    p = r.choice(pos); return text[:p] + DIAC_SWAP[text[p]] + text[p + 1:], k
        return None, None

    def _fabricate(self, label):
        a, b = self._pick(label)['text'].split(), self._pick(label)['text'].split()
        return ' '.join(a[:max(2, len(a) // 2)] + b[len(b) // 2:])

    def _make_quote(self, label):
        r = self.rng
        item = self._multi_ayah() if (label == 'Ayah' and r.random() < 0.12) else self._pick(label)
        u = r.random()
        if u < 0.50 or item['multi']:
            return item['text'], 'Correct', None, item['ref']
        if u < 0.80:
            bad, kind = self._corrupt(item['text'], label)
            if bad:
                fix = apply_idgham(item['text']) if label == 'Ayah' else item['text']
                return bad, 'Incorrect', (fix, kind), item['ref']
        return self._fabricate(label), 'Incorrect', ('خطأ', 'fabricated'), None

    @staticmethod
    def _wrap(label, r):
        if label == 'Ayah':
            return r.choices([('"', '"'), ('(', ')'), ('{', '}'), ('', '')], [0.55, 0.15, 0.15, 0.15])[0]
        return r.choices([('"', '"'), ('«', '»'), ('(', ')'), ('', '')], [0.75, 0.08, 0.07, 0.10])[0]

    def make_one(self, idx):
        r = self.rng
        topic = r.choice(TOPICS)
        text, spans = r.choice(INTROS).format(t=topic) + ' ', []
        if r.random() < 0.08:                                       
            text += ' '.join(r.sample(NEGATIVES[:4], 2))
            return {'id': f'SYN-{idx:05d}', 'source': 'synthetic', 'text': text, 'spans': [],
                    'complete': {'detection': True, 'verdict': True, 'correction': True}}
        for _ in range(r.choices([1, 2, 3, 4, 5], [0.25, 0.30, 0.25, 0.12, 0.08])[0]):
            label = r.choice(['Ayah', 'Hadith'])
            quote, verdict, fix, ref = self._make_quote(label)
            op, cl = self._wrap(label, r)
            lead = r.choice(AYAH_LEADS if label == 'Ayah' else HADITH_LEADS)
            text += lead + ' ' + op
            s = len(text)
            text += quote
            spans.append(_span(s, len(text), label, quote, verdict=verdict, correction=fix[0] if fix else None,
                               corruption=fix[1] if fix else None))
            text += cl
            if cl == '':
                text += '.'
            if label == 'Ayah' and ref and r.random() < 0.35:
                text += f' [{ref}]'
            text += ' ' + r.choice(TAILS) + ('\n' if r.random() < 0.3 else ' ')
            if r.random() < 0.25:
                text += r.choice(NEGATIVES) + ' '
        return {'id': f'SYN-{idx:05d}', 'source': 'synthetic', 'text': text.rstrip(), 'spans': spans,
                'complete': {'detection': True, 'verdict': True, 'correction': True}}


def build_synthetic_records(retriever, n=1500, seed=42):
    b = SyntheticBuilder(retriever, seed)
    out = [b.make_one(i + 1) for i in range(n)]
    for rec in out:                                                   
        for s in rec['spans']:
            assert rec['text'][s['start']:s['end']] == s['text']
    return out


def build_unified_dataset(retriever, data_dir='.', n_synthetic=1500, seed=42, out_path='islamic_unified_dataset.jsonl'):
    recs = build_real_records(data_dir) + build_synthetic_records(retriever, n_synthetic, seed)
    with open(out_path, 'w', encoding='utf-8') as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    return recs


def load_unified_dataset(path='islamic_unified_dataset.jsonl'):
    return [json.loads(l) for l in open(path, encoding='utf-8')]


def dataset_stats(recs):
    from collections import Counter
    c = Counter(r['source'] for r in recs)
    print(f'TOTAL records: {len(recs)}   ->   ' + '   '.join(f'{k}: {v}' for k, v in sorted(c.items())))
    for src in sorted(c):
        sp = [s for r in recs if r['source'] == src for s in r['spans']]
        v = Counter((s['label'], s['verdict']) for s in sp)
        print(f'  {src:<10} spans={len(sp):5d}  ' + '  '.join(f'{a}/{b}={n}' for (a, b), n in sorted(v.items(), key=str)))


def get_dataset(retriever, n_synthetic=1500, path='islamic_unified_dataset.jsonl', data_dir='.', seed=42):
    import os
    if os.path.exists(path):
        recs = load_unified_dataset(path)
        real = [r for r in recs if r['source'].startswith('real')]
        if n_synthetic != sum(r['source'] == 'synthetic' for r in recs):
            recs = real + build_synthetic_records(retriever, n_synthetic, seed)
            with open(path, 'w', encoding='utf-8') as f:
                for r in recs:
                    f.write(json.dumps(r, ensure_ascii=False) + '\n')
        return recs
    return build_unified_dataset(retriever, data_dir, n_synthetic, seed, path)


kb = SourceRetriever('quranic_verses.json', 'six_hadith_books.json', cache_path='.cache/idx.pkl')
N_SYNTHETIC = 1500                      
records = get_dataset(kb, n_synthetic=N_SYNTHETIC)
dataset_stats(records)

In [ ]:
def detection_items(records, use_B=True, exclude_ids=()):
    keep = {'real_A', 'synthetic'} | ({'real_B'} if use_B else set())
    return [(r['id'], r['text'], [{'label': s['label'], 'start': s['start'], 'end': s['end']} for s in r['spans']])
            for r in records if r['source'] in keep and r['complete']['detection'] and r['id'] not in set(exclude_ids)]


def train_detector(records, out_dir='models/span_detector', model_name='aubmindlab/bert-base-arabertv2',
                   epochs=8, batch=4, lr=3e-5, use_B=True, exclude_ids=(), seed=42, max_length=512, stride=256):
    import numpy as np, torch
    from torch.utils.data import DataLoader, Dataset
    from transformers import AutoModelForTokenClassification, AutoTokenizer, get_linear_schedule_with_warmup
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    items = detection_items(records, use_B, exclude_ids)
    print(f'training on {len(items)} texts with {model_name} on {dev}')
    tok = AutoTokenizer.from_pretrained(model_name)

    class Windows(Dataset):
        def __init__(self):
            self.w = []
            for _, text, anns in items:
                cl = build_character_labels(text, anns)
                enc = tok(text, return_offsets_mapping=True, truncation=False)
                ids, mask, offs = enc['input_ids'], enc['attention_mask'], enc['offset_mapping']
                st = 0
                while st < len(ids):
                    en = min(st + max_length, len(ids)); pad = max_length - (en - st)
                    wo = offs[st:en] + [(0, 0)] * pad
                    self.w.append({'input_ids': torch.tensor(ids[st:en] + [tok.pad_token_id] * pad),
                                   'attention_mask': torch.tensor(mask[st:en] + [0] * pad),
                                   'labels': torch.tensor(char_labels_to_token_labels(wo, cl))})
                    if en == len(ids): break
                    st += stride
        def __len__(self): return len(self.w)
        def __getitem__(self, i): return self.w[i]

    model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=len(LABEL2ID), id2label=ID2LABEL, label2id=LABEL2ID).to(dev)
    loader = DataLoader(Windows(), batch_size=batch, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    total = len(loader) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)
    model.train()
    for ep in range(epochs):
        run = 0.0
        for b in loader:
            opt.zero_grad()
            loss = model(**{k: v.to(dev) for k, v in b.items()}).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); run += loss.item()
        print(f'  epoch {ep + 1}/{epochs}  loss {run / len(loader):.4f}')
    model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    print('saved ->', out_dir)
    return out_dir


def evaluate_detector(detector, records, ids):
    """Official 1A macro-F1 on the given real_A record ids."""
    recs = [r for r in records if r['id'] in set(ids)]
    resp = {r['id']: r['text'] for r in recs}
    gold = {r['id']: [{'label': s['label'], 'start': s['start'], 'end': s['end']} for s in r['spans']] for r in recs}
    pred = {q: [{'label': d.label, 'start': d.start, 'end': d.end} for d in detector.detect(t)] for q, t in resp.items()}
    return detection_score(resp, gold, pred)


RUN_CHECK = True                                              
if RUN_CHECK:
    HOLDOUT = [r['id'] for r in records if r['source'] == 'real_A'][:10]
    train_detector(records, 'models/check_model', exclude_ids=HOLDOUT)
    print('F1 على الـ10 المخبيين  AraBERT:', round(evaluate_detector(BertDetector('models/check_model'), records, HOLDOUT), 4),
          '| القواعد:', round(evaluate_detector(RuleDetector(None), records, HOLDOUT), 4))

In [ ]:
train_detector(records, 'models/span_detector')        

In [ ]:
import os


def _clean(s):
    return str(s).replace('\t', ' ').replace('\r', ' ').replace('\n', ' ').strip()


def competition_rows(result, qid):
    a, b, c = [], [], []
    spans = result['spans']
    if not spans:
        a.append((qid, 0, 0, 'No_Spans'))
    for k, sp in enumerate(spans, 1):
        a.append((qid, sp['start'], sp['end'], sp['type']))
        b.append((f'{qid}_{k}', sp['verification']['verdict']))
        if sp['verification']['verdict'] == 'Incorrect':
            corr = sp['correction']['text'] if (sp['status'] == 'CORRECTED' and sp['correction']) else 'خطأ'
            c.append((f"{qid}_{sp['start']}_{sp['end']}", _clean(corr)))
    return a, b, c


def write_competition_files(all_rows, out_dir='outputs'):
    os.makedirs(out_dir, exist_ok=True)
    names = ['task1A_predictions.tsv', 'task1B_predictions.tsv', 'task1C_predictions.tsv']
    for name, rows in zip(names, all_rows):
        with open(os.path.join(out_dir, name), 'w', encoding='utf-8', newline='') as f:
            for r in rows:
                f.write('\t'.join(str(x) for x in r) + '\n')
    return [os.path.join(out_dir, n) for n in names]


def run_on_texts(pipe, texts, out_dir='outputs', show=True):
    rows, results = ([], [], []), {}
    for qid, text in texts.items():
        res = pipe.analyze(text)
        results[qid] = res
        for acc, part in zip(rows, competition_rows(res, qid)):
            acc.extend(part)
        if show:
            print(f'\n######## {qid} ########')
            print(render_report(res))
    paths = write_competition_files(rows, out_dir)
    with open(os.path.join(out_dir, 'full_report.json'), 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print('\nfiles written:', *paths, os.path.join(out_dir, 'full_report.json'), sep='\n  ')
    return results, paths


def run_on_xml(pipe, xml_path, out_dir='outputs', show=False):
    return run_on_texts(pipe, load_responses(xml_path), out_dir, show)


def build_pipeline(quran='quranic_verses.json', hadith='six_hadith_books.json', model_dir='models/span_detector'):
    has_model = os.path.isdir(model_dir)
    pipe = IslamicContentVerifier(quran, hadith, detector='auto', model_dir=model_dir if has_model else None, verbose=False)
    print('detector in use:', pipe.detector_name, '' if has_model else '(no trained model found -> rule-based fallback)')
    return pipe

def make_examples(pipe):
    kb, r = pipe.retriever, pipe.corrector
    ay = lambda s, a: kb.quran[kb.quran_by_surah[s][a]]['text']
    had = kb.search_hadith('إنما الأعمال بالنيات وإنما لكل امرئ ما نوى', top_k=1)[0]['text']
    had2 = kb.search_hadith('الراحمون يرحمهم الرحمن ارحموا من في الأرض يرحمكم من في السماء', top_k=1)[0]['text']
    a1, a2 = ay(2, 153), ay(17, 23)
    w = a1.split(); a1_bad = ' '.join(w[:2] + w[3:])                     
    fake = ' '.join(had.split()[:len(had.split()) // 2] + had2.split()[len(had2.split()) // 2:])
    return [
        ('المستوى 1: نص بدون أي اقتباس (المتوقع: No_Spans)',
         'الصبر من أهم الصفات التي ينبغي أن يتحلى بها الإنسان في حياته، ويساعده على مواجهة المصاعب بثبات وهدوء.'),
        ('المستوى 2: آية صحيحة بين علامات تنصيص (المتوقع: VERIFIED)',
         f'يقول الله تعالى: "{a1}". وهذه الآية تدل على معية الله للصابرين.'),
        ('المستوى 3: حديث صحيح (المتوقع: VERIFIED)',
         f'وقال رسول الله صلى الله عليه وسلم: "{had}". ويستفاد منه أهمية النية في الأعمال.'),
        ('المستوى 4: آية فيها كلمة ناقصة (المتوقع: HUMAN_REVIEW مع اقتراح الآية الصحيحة)',
         f'قال الله تعالى: "{a1_bad}".'),
        ('المستوى 5: حديث مُختلَق بالكامل (المتوقع: UNSUPPORTED)',
         'وقال رسول الله صلى الله عليه وسلم: "من أكل التفاح كل يوم زاد رزقه وطال عمره وصرف الله عنه كل مصيبة وبلاء".'),
        ('المستوى 6: رد طويل مختلط (آية صحيحة + حديث صحيح + آية ناقصة كلمة + حديث نصفه من حديثين مختلفين)',
         f'بخصوص بر الوالدين، قال الله تعالى: "{a2}". وقال رسول الله صلى الله عليه وسلم: "{had2}". '
         f'وقال سبحانه: "{a1_bad}". وروى البخاري أن النبي صلى الله عليه وسلم قال: "{fake}". وهذا من مكارم الأخلاق.'),
        ('المستوى 7: أشكال كتابة مختلفة للاقتباس: ( ) و { } و " "',
         f'وقال تعالى: ({a1}) وقال الله تعالى: {{{a2}}} وقال رسول الله صلى الله عليه وسلم: "{had}".'),
        ('المستوى 8: اقتباس بدون أي علامات تنصيص (يحتاج موديل AraBERT، كاشف القواعد غالبًا هيفوّته)',
         f'وجاء في الحديث الشريف عن النبي صلى الله عليه وسلم {had2} وهذا يدل على الرحمة بالخلق.'),
    ]


pipe = build_pipeline()

In [ ]:
USER_TEXT = """"""    
if not USER_TEXT.strip():
    USER_TEXT = input('الصقي النص ثم Enter: ')
results, paths = run_on_texts(pipe, {'U-001': USER_TEXT}, out_dir='outputs')
print(open('outputs/task1A_predictions.tsv', encoding='utf-8').read())
print(open('outputs/task1B_predictions.tsv', encoding='utf-8').read())
print(open('outputs/task1C_predictions.tsv', encoding='utf-8').read())

In [ ]:
from google.colab import files
import shutil
up = files.upload()
xml_name = next(iter(up))
results, paths = run_on_xml(pipe, xml_name, out_dir='outputs_batch')
shutil.make_archive('outputs_batch', 'zip', 'outputs_batch')
files.download('outputs_batch.zip')

In [ ]:
ex = make_examples(pipe)
results, _ = run_on_texts(pipe, {f'EX-{i + 1}': t for i, (title, t) in enumerate(ex)}, out_dir='outputs_examples', show=False)
for (title, text), (qid, res) in zip(ex, results.items()):
    print('\n' + '#' * 100 + '\n' + title + '\n' + '#' * 100)
    print('الإدخال :', text[:300])
    print(render_report(res))

In [ ]:
def evaluate_dev(pipe, records):
    B = [(r['text'][s['start']:s['end']].strip(), s['label'], s['verdict']) for r in records if r['source'] == 'real_B' for s in r['spans']]
    ok = sum(pipe.verifier.verify(t, lab).verdict == v for t, lab, v in B)
    print(f'[1B] verification accuracy on gold spans : {ok / len(B):.4f} ({ok}/{len(B)})   (all-"Correct" baseline = {sum(v == "Correct" for _, _, v in B) / len(B):.3f})')
    C = [(r['text'][s['start']:s['end']], s['label'], s['correction']) for r in records if r['source'] == 'real_C' for s in r['spans']]
    sc, cc, ok = CorrectionScorer('quranic_verses.json', 'six_hadith_books.json'), pipe.cfg.corrector, 0
    for t, lab, corr in C:
        m = pipe.corrector.match(t, lab)
        strong = cc.quran_strong if lab == 'Ayah' else cc.hadith_strong
        out = m.text if (m and m.strength >= strong and (lab != 'Ayah' or m.full_ratio >= cc.min_full_ratio)) else 'خطأ'
        ok += sc.is_correct(out, corr)
    print(f'[1C] correction accuracy (official rule)  : {ok / len(C):.4f} ({ok}/{len(C)})   (all-"خطأ" baseline = {sum(c == "خطأ" for _, _, c in C) / len(C):.3f})')
    print(f'[1A] detection F1 on real_A ({pipe.detector_name}) : {evaluate_detector(pipe.detector, records, [r["id"] for r in records if r["source"] == "real_A"]):.4f}   <- inflated if these questions were in the training set')


evaluate_dev(pipe, records)